# End-to-end Kaggle retraining

Practical offline retraining path for this repo:

```text
PCAP/PCAPNG -> payload_256.npy + metadata.csv -> teacher embeddings -> student CNN -> student_embeddings.npy -> MITRE embeddings -> 3-tier graph NPZ -> mmap/CSR graph store -> HGT neighbor-sampling train
```

Training from one packet is not valid; runtime can infer one packet, but offline HGT retraining needs a labeled packet/flow corpus. Current extractor infers label from PCAP filename prefix, e.g. `Recon-PortScan.pcap` -> `Recon`.


In [ ]:
from __future__ import annotations
import csv, glob, json, math, os, queue, shutil, subprocess, sys, zipfile
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from typing import Any
import torch

PIPELINE_MODE = "full_from_pcap"       # full_from_pcap | existing_graph_npz
HGT_RUN_MODE = "deployment"            # deployment | paper_variants
GITHUB_REPO_URL = "https://github.com/LeThanhPhat-ATTT2023/Do-an-chuyen-nganh_NT114.git"
GITHUB_BRANCH = ""
WORK_DIR = Path("/kaggle/working/nt114_hgt_work")
RESULT_ZIP = Path("/kaggle/working/hgt_training_results_kaggle.zip")
RAW_PCAP_GLOBS = ["/kaggle/input/**/*.pcap", "/kaggle/input/**/*.pcapng"]

PAYLOAD_LENGTH = 256
MAX_PACKETS_PER_FILE = None
INCLUDE_EMPTY_PAYLOAD = False
FLOW_TIMEOUT_SECONDS = 30.0
MAX_PACKETS_PER_FLOW = 20
SIMILARITY_THRESHOLD = 0.82
PACKET_TOP_K = 5
FLOW_TOP_K = 5

TEACHER_MODEL_NAME = "ehsanaghaei/SecureBERT"
TEACHER_BATCH_SIZE = 32
TEACHER_MAX_LENGTH = 512
STUDENT_EPOCHS = 30
STUDENT_BATCH_SIZE = 256
STUDENT_NUM_WORKERS = 2
STUDENT_EMB_BATCH_SIZE = 1024
MITRE_BATCH_SIZE = 64

GRAPH_NPZ_NAME = "graph_artifact_3tier_t082_k5.npz"
GRAPH_META_NAME = "graph_artifact_3tier_t082_k5.meta.json"
TRAIN_GRAPH_STORE_DIR_NAME = "graph_store_mmap_csr_v1"
GPU_IDS = [0, 1]
MAX_PARALLEL_HGT_RUNS = 2
SAFE_15GB_PROFILE = True
MAX_BATCH_SEED_FLOWS = 128
MIN_BATCH_SEED_FLOWS = 16
OOM_RETRIES = 3
INSTALL_MISSING_DEPS = True
INSTALL_WITH_DEPS = False
RESET_OUTPUTS = False
USE_INPUT_TRAIN_GRAPH_STORE_IF_PRESENT = True
COPY_LARGE_INPUT_ARTIFACTS_TO_WORK_DIR = False

DEPLOYMENT_RUNS = [{"name":"deployment_t082_k5_l3_d01","config":"configs/hgt_t082_k5_l3_d01.yaml"}]
PAPER_VARIANT_RUNS = [
 {"name":"baseline_t082_k5_l3_d01","config":"configs/hgt_t082_k5_l3_d01.yaml"},
 {"name":"xgnid_dual_modal_l1_h32_h4","config":"configs/hgt_paper_variants/hgt_t082_k5_xgnid_dual_modal_l1_h32_h4.yaml"},
 {"name":"one2_iov_l1_h64_h2","config":"configs/hgt_paper_variants/hgt_t082_k5_one2_iov_l1_h64_h2.yaml"},
 {"name":"relgt_multi_token_l3_h128_h8","config":"configs/hgt_paper_variants/hgt_t082_k5_relgt_multi_token_l3_h128_h8.yaml"},
 {"name":"gatransformer_deep_l6_h256_h8","config":"configs/hgt_paper_variants/hgt_t082_k5_gatransformer_deep_l6_h256_h8.yaml"},
 {"name":"ahgt_dfd_funnel_l3_h128_h4","config":"configs/hgt_paper_variants/hgt_t082_k5_ahgt_dfd_funnel_l3_h128_h4.yaml"},
 {"name":"dlg_ids_sparse_l2_h128_h4","config":"configs/hgt_paper_variants/hgt_t082_k5_dlg_ids_sparse_l2_h128_h4.yaml"},
]
RUNS = DEPLOYMENT_RUNS if HGT_RUN_MODE == "deployment" else PAPER_VARIANT_RUNS


In [ ]:
print("Torch:", torch.__version__, "CUDA:", torch.cuda.is_available(), "GPU count:", torch.cuda.device_count())
if not torch.cuda.is_available():
    raise RuntimeError("Enable Kaggle GPU first: Settings -> Accelerator -> GPU T4 x2.")
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {p.name}, VRAM={p.total_memory/1024**3:.1f} GB")
GPU_IDS = [i for i in GPU_IDS if i < torch.cuda.device_count()] or [0]
MAX_PARALLEL_HGT_RUNS = min(MAX_PARALLEL_HGT_RUNS, len(GPU_IDS), len(RUNS))
print("HGT_RUN_MODE:", HGT_RUN_MODE, "GPU_IDS:", GPU_IDS, "parallel HGT:", MAX_PARALLEL_HGT_RUNS)


In [ ]:
def run(cmd, cwd=None, env=None):
    print("\n$", " ".join(map(str, cmd)))
    subprocess.check_call([str(x) for x in cmd], cwd=cwd or WORK_DIR, env=env)

def is_repo(p: Path) -> bool:
    return (p/"src/graphslm_ids").exists() and (p/"configs/hgt_t082_k5_l3_d01.yaml").exists()

def input_file(name: str) -> Path | None:
    roots = [WORK_DIR/"data/processed", WORK_DIR/"data/mitre", Path("/kaggle/input")]
    for r in roots:
        if not r.exists(): continue
        direct = r/name
        if direct.exists(): return direct
        for m in r.rglob(name): return m
    return None

def copy_or_use(name: str, target: Path, *, required=True, copy=True) -> Path | None:
    if target.exists(): return target
    src = input_file(name)
    if src is None:
        if required: raise FileNotFoundError(f"Missing {name}")
        return None
    if not copy: return src
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, target)
    return target

def find_repo_input() -> Path | None:
    root = Path("/kaggle/input")
    if not root.exists(): return None
    for p in sorted(root.glob("*")):
        if is_repo(p): return p
        for cfg in p.rglob("configs/hgt_t082_k5_l3_d01.yaml"):
            cand = cfg.parents[1]
            if is_repo(cand): return cand
    return None

def prepare_repo():
    if is_repo(WORK_DIR): return
    if WORK_DIR.exists(): shutil.rmtree(WORK_DIR)
    src = find_repo_input()
    if src:
        shutil.copytree(src, WORK_DIR, ignore=shutil.ignore_patterns(".git","__pycache__","*.pyc",".pytest_cache")); return
    if not GITHUB_REPO_URL: raise RuntimeError("Add repo Kaggle Dataset or set GITHUB_REPO_URL")
    cmd=["git","clone"] + (["--branch",GITHUB_BRANCH] if GITHUB_BRANCH else []) + [GITHUB_REPO_URL, str(WORK_DIR)]
    subprocess.check_call(cmd)

def import_ok(m):
    try: __import__(m); return True
    except Exception: return False

def install():
    miss=[m for m in ["numpy","pandas","scapy","tqdm","yaml","transformers"] if not import_ok(m)]
    if miss and INSTALL_MISSING_DEPS: run([sys.executable,"-m","pip","install","-r","requirements-ml.txt"])
    elif miss: raise RuntimeError(f"Missing deps: {miss}")
    cmd=[sys.executable,"-m","pip","install","-e","."] + ([] if INSTALL_WITH_DEPS else ["--no-deps"])
    run(cmd)

prepare_repo(); os.chdir(WORK_DIR)
if RESET_OUTPUTS and (WORK_DIR/"outputs").exists(): shutil.rmtree(WORK_DIR/"outputs")
install()

PAYLOAD_DIR=WORK_DIR/"data/interim/payload_dataset"; PAYLOAD_NPY=PAYLOAD_DIR/"payload_256.npy"; METADATA_CSV=PAYLOAD_DIR/"metadata.csv"
PROCESSED=WORK_DIR/"data/processed"; TEACHER_NPY=PROCESSED/"teacher_targets.npy"; STUDENT_EMB_NPY=PROCESSED/"student_embeddings.npy"
GRAPH_NPZ=PROCESSED/GRAPH_NPZ_NAME; GRAPH_META_JSON=PROCESSED/GRAPH_META_NAME
STUDENT_DIR=WORK_DIR/"outputs/student_cnn"; STUDENT_CKPT=STUDENT_DIR/"student_cnn_best.pt"
MITRE=WORK_DIR/"data/mitre"; MITRE_STIX=MITRE/"enterprise-attack.json"; MITRE_TECH=MITRE/"mitre_techniques.csv"
MITRE_EDGE=MITRE/"mitre_technique_tactic_edges.csv"; MITRE_TAC=MITRE/"mitre_tactics.csv"; MITRE_EMB=MITRE/"mitre_techniques_embeddings.npy"
STORE_ROOT=WORK_DIR/"data"/TRAIN_GRAPH_STORE_DIR_NAME
RUNTIME_CONFIG_DIR=WORK_DIR/"outputs/kaggle_runtime_configs"; LOG_DIR=WORK_DIR/"outputs/hgt_kaggle_logs"
RUNTIME_CONFIG_DIR.mkdir(parents=True, exist_ok=True); LOG_DIR.mkdir(parents=True, exist_ok=True)
print("WORK_DIR:", WORK_DIR)


In [ ]:
def stage_payload():
    if PIPELINE_MODE != "full_from_pcap" or (PAYLOAD_NPY.exists() and METADATA_CSV.exists()): return
    pcaps = sorted({Path(x) for pat in RAW_PCAP_GLOBS for x in glob.glob(pat, recursive=True) if Path(x).suffix.lower() in {".pcap",".pcapng"}})
    if not pcaps: raise RuntimeError("No PCAP/PCAPNG found. Add raw traffic dataset or edit RAW_PCAP_GLOBS.")
    cmd=[sys.executable,"-u","-m","graphslm_ids.offline_path.preprocessing.extract_payload_dataset","--input-glob",*RAW_PCAP_GLOBS,"--output-dir",str(PAYLOAD_DIR),"--payload-length",str(PAYLOAD_LENGTH),"--export-graph-csv","--graph-flow-timeout-seconds",str(FLOW_TIMEOUT_SECONDS),"--graph-max-packets-per-flow",str(MAX_PACKETS_PER_FLOW)]
    if MAX_PACKETS_PER_FILE is not None: cmd += ["--max-packets-per-file", str(MAX_PACKETS_PER_FILE)]
    if INCLUDE_EMPTY_PAYLOAD: cmd += ["--include-empty-payload"]
    run(cmd)

def stage_teacher_student():
    if PIPELINE_MODE != "full_from_pcap": return
    if not TEACHER_NPY.exists():
        run([sys.executable,"-u","-m","graphslm_ids.offline_path.preprocessing.build_teacher_targets","--payload-npy",str(PAYLOAD_NPY),"--metadata-csv",str(METADATA_CSV),"--output-path",str(TEACHER_NPY),"--model-name",TEACHER_MODEL_NAME,"--batch-size",str(TEACHER_BATCH_SIZE),"--max-length",str(TEACHER_MAX_LENGTH),"--device","cuda:0"])
    if not STUDENT_CKPT.exists():
        run([sys.executable,"-u","-m","graphslm_ids.offline_path.training.train_student_cnn","--payload-npy",str(PAYLOAD_NPY),"--teacher-npy",str(TEACHER_NPY),"--output-dir",str(STUDENT_DIR),"--batch-size",str(STUDENT_BATCH_SIZE),"--epochs",str(STUDENT_EPOCHS),"--num-workers",str(STUDENT_NUM_WORKERS),"--device","cuda:0"])
    if not STUDENT_EMB_NPY.exists():
        run([sys.executable,"-u","-m","graphslm_ids.offline_path.training.export_student_embeddings","--payload-npy",str(PAYLOAD_NPY),"--checkpoint",str(STUDENT_CKPT),"--output-path",str(STUDENT_EMB_NPY),"--batch-size",str(STUDENT_EMB_BATCH_SIZE),"--device","cuda:0"])

def stage_mitre():
    if PIPELINE_MODE != "full_from_pcap": return
    copy_or_use("mitre_techniques.csv", MITRE_TECH, required=False); copy_or_use("mitre_technique_tactic_edges.csv", MITRE_EDGE, required=False); copy_or_use("mitre_tactics.csv", MITRE_TAC, required=False)
    if not (MITRE_TECH.exists() and MITRE_EDGE.exists()):
        copy_or_use("enterprise-attack.json", MITRE_STIX, required=True)
        run([sys.executable,"-u","-m","graphslm_ids.offline_path.preprocessing.prepare_mitre_knowledge_base","--input-json",str(MITRE_STIX),"--techniques-csv",str(MITRE_TECH),"--tactics-csv",str(MITRE_TAC),"--technique-tactic-edges-csv",str(MITRE_EDGE),"--stats-json",str(MITRE/"mitre_export_stats.json")])
    if not MITRE_EMB.exists():
        run([sys.executable,"-u","-m","graphslm_ids.offline_path.preprocessing.build_mitre_technique_embeddings","--techniques-csv",str(MITRE_TECH),"--output-path",str(MITRE_EMB),"--teacher-meta-json",str(TEACHER_NPY.with_suffix(".meta.json")),"--batch-size",str(MITRE_BATCH_SIZE),"--device","cuda:0"])

def stage_graph_npz():
    global GRAPH_NPZ, GRAPH_META_JSON
    if PIPELINE_MODE == "existing_graph_npz":
        g=copy_or_use(GRAPH_NPZ_NAME, GRAPH_NPZ, copy=COPY_LARGE_INPUT_ARTIFACTS_TO_WORK_DIR); m=copy_or_use(GRAPH_META_NAME, GRAPH_META_JSON, copy=True)
        if g != GRAPH_NPZ: GRAPH_NPZ = g
        if m != GRAPH_META_JSON: GRAPH_META_JSON = m
        return
    if GRAPH_NPZ.exists() and GRAPH_META_JSON.exists(): return
    run([sys.executable,"-u","-m","graphslm_ids.offline_path.preprocessing.build_three_tier_graph_artifact","--metadata-csv",str(METADATA_CSV),"--payload-npy",str(PAYLOAD_NPY),"--student-embedding-npy",str(STUDENT_EMB_NPY),"--mitre-techniques-csv",str(MITRE_TECH),"--mitre-technique-embeddings-npy",str(MITRE_EMB),"--mitre-technique-tactic-edges-csv",str(MITRE_EDGE),"--output-npz",str(GRAPH_NPZ),"--output-meta-json",str(GRAPH_META_JSON),"--flow-timeout-seconds",str(FLOW_TIMEOUT_SECONDS),"--max-packets-per-flow",str(MAX_PACKETS_PER_FLOW),"--similarity-threshold",str(SIMILARITY_THRESHOLD),"--packet-top-k",str(PACKET_TOP_K),"--flow-top-k",str(FLOW_TOP_K)])

stage_payload(); stage_teacher_student(); stage_mitre(); stage_graph_npz()
print("Graph NPZ:", GRAPH_NPZ)


In [ ]:
def layout(root: Path) -> str | None:
    p=root/"manifest.json"
    if not p.exists(): return None
    return str(json.loads(p.read_text()).get("layout",""))

def is_train_store(root: Path) -> bool: return layout(root) == "numpy_memmap_csr"

def input_train_store() -> Path | None:
    if not Path("/kaggle/input").exists(): return None
    for m in sorted(Path("/kaggle/input").rglob("manifest.json")):
        if is_train_store(m.parent): return m.parent
    return None

def stage_store() -> Path:
    if is_train_store(STORE_ROOT): return STORE_ROOT
    legacy=WORK_DIR/"data/graph_store_v1"
    if layout(legacy) and layout(legacy)!="numpy_memmap_csr": print("Ignore runtime graph_store_v1 layout:", layout(legacy))
    src=input_train_store() if USE_INPUT_TRAIN_GRAPH_STORE_IF_PRESENT else None
    if src: return src
    run([sys.executable,"-u","-m","graphslm_ids.offline_path.training.on_disk_graph_store","--graph-npz",str(GRAPH_NPZ),"--graph-meta-json",str(GRAPH_META_JSON),"--output-root",str(STORE_ROOT)])
    if not is_train_store(STORE_ROOT): raise RuntimeError("Graph store layout must be numpy_memmap_csr")
    return STORE_ROOT

GRAPH_STORE_ROOT = stage_store()
print("GRAPH_STORE_ROOT:", GRAPH_STORE_ROOT)
print("GRAPH_STORE_LAYOUT:", layout(GRAPH_STORE_ROOT))


In [ ]:
import yaml

def yload(p: Path) -> dict[str, Any]:
    return yaml.safe_load(p.read_text(encoding="utf-8")) or {}

def ydump(p: Path, d: dict[str, Any]):
    p.parent.mkdir(parents=True, exist_ok=True); p.write_text(yaml.safe_dump(d, sort_keys=False), encoding="utf-8")

def patch_cfg(run_cfg, retry=0):
    cfg=yload(WORK_DIR/run_cfg["config"])
    data=cfg.setdefault("data",{}); data.update(source="graph_store", graph_npz=str(GRAPH_NPZ), graph_meta_json=str(GRAPH_META_JSON), graph_store_root=str(GRAPH_STORE_ROOT), read_sealed_only=True, packet_feature=data.get("packet_feature","semantic"), add_reverse_edges=True, standardize_flow_features=True, use_semantic_edge_weights=True)
    tr=cfg.setdefault("train",{}); tr.update(batch_mode="neighbor_sampling", device="cuda", amp=True, activation_checkpointing=True, monitor=tr.get("monitor","val_macro_f1"), log_every=tr.get("log_every",1))
    base=int(tr.get("batch_seed_flows",256)); grad=int(tr.get("grad_accum_steps",1)); cap=min(base,MAX_BATCH_SEED_FLOWS) if SAFE_15GB_PROFILE else base; bs=max(MIN_BATCH_SEED_FLOWS, cap//(2**retry))
    tr["batch_seed_flows"]=bs; tr["grad_accum_steps"]=max(grad, math.ceil((base*grad)/bs))
    smp=cfg.setdefault("sampler",{}); smp.setdefault("hops",None); smp.setdefault("fanouts",{"flow__contains__packet":20,"packet__next_packet__packet":4,"packet__matches_technique__technique":5,"flow__matches_technique__technique":5,"technique__belongs_to_tactic__tactic":1}); smp.setdefault("reverse_fanouts",{"rev_contains":1,"rev_next_packet":1,"rev_matches_technique":0,"rev_belongs_to_tactic":0}); smp.setdefault("always_include_all_tactics",True); smp.setdefault("always_include_all_techniques",True)
    dl=cfg.setdefault("dataloader",{}); dl.update(num_workers=min(int(dl.get("num_workers",2)),2), prefetch_factor=int(dl.get("prefetch_factor",2)), pin_memory=True)
    out=RUNTIME_CONFIG_DIR/(run_cfg["name"]+("" if retry==0 else f"_retry{retry}")+".yaml"); ydump(out,cfg); return out,cfg

def summary_path(cfg_path: Path) -> Path: return WORK_DIR/yload(cfg_path)["train"]["output_dir"]/"training_summary.json"
def oom(log: Path) -> bool: return log.exists() and any(x in log.read_text(errors="ignore") for x in ["CUDA out of memory","torch.OutOfMemoryError","CUBLAS_STATUS_ALLOC_FAILED"])

def train_proc(run_cfg, cfg_path, gpu, retry):
    dev=f"gpu{gpu}"; log=LOG_DIR/f"{run_cfg['name']}_{dev}_retry{retry}.log"; env=os.environ.copy(); env.update(PYTHONUNBUFFERED="1", PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True,max_split_size_mb:128", CUDA_VISIBLE_DEVICES=str(gpu), OMP_NUM_THREADS="2", MKL_NUM_THREADS="2")
    cmd=[sys.executable,"-u","-m","graphslm_ids.offline_path.training.train_hgt_flow_classifier","--config",str(cfg_path),"--device","cuda"]
    print("\nTRAIN", run_cfg["name"], "on", dev, "log", log)
    with log.open("w", encoding="utf-8") as f:
        p=subprocess.Popen(cmd,cwd=WORK_DIR,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,env=env)
        for line in p.stdout: print(f"[{dev}:{run_cfg['name']}] {line}", end=""); f.write(line); f.flush()
        return p.wait(), log

def train_one(run_cfg, gpu):
    cfg0,_=patch_cfg(run_cfg,0); s=summary_path(cfg0)
    if s.exists(): return {"run":run_cfg["name"],"status":"skipped","summary":str(s)}
    last=None
    for r in range(OOM_RETRIES+1):
        cfg,_=patch_cfg(run_cfg,r); code,log=train_proc(run_cfg,cfg,gpu,r); last=log
        if code==0: return {"run":run_cfg["name"],"status":"ok","summary":str(summary_path(cfg))}
        if not oom(log) or r==OOM_RETRIES: break
    raise RuntimeError(f"HGT run failed: {run_cfg['name']}. See {last}")

def train_all():
    q=queue.Queue(); [q.put(g) for g in GPU_IDS[:MAX_PARALLEL_HGT_RUNS]]; out=[]
    def worker(r):
        g=q.get()
        try: return train_one(r,g)
        finally: q.put(g)
    with ThreadPoolExecutor(max_workers=MAX_PARALLEL_HGT_RUNS) as ex:
        for fut in as_completed([ex.submit(worker,r) for r in RUNS]): out.append(fut.result()); print("DONE", out[-1])
    return out

hgt_results=train_all(); hgt_results


In [ ]:
def build_comparison():
    rows=[]
    for r in RUNS:
        cfg,_=patch_cfg(r,0); sp=summary_path(cfg)
        if not sp.exists(): continue
        d=json.loads(sp.read_text()); c=d["config"]; m=c["model"]; t=c["train"]; bv=d.get("best_val_metrics",{}); bt=d.get("best_test_metrics",{})
        rows.append({"run_name":r["name"],"run_dir":str(sp.parent.relative_to(WORK_DIR)),"hidden_dim":m["hidden_dim"],"num_layers":m["num_layers"],"num_heads":m["num_heads"],"batch_mode":t.get("batch_mode"),"batch_seed_flows":t.get("batch_seed_flows"),"grad_accum_steps":t.get("grad_accum_steps"),"best_epoch":d.get("best_epoch"),"val_macro_f1":bv.get("macro_f1"),"test_macro_f1":bt.get("macro_f1"),"test_accuracy":bt.get("accuracy"),"device":d.get("device")})
    out=WORK_DIR/"outputs"; out.mkdir(exist_ok=True); csvp=out/"hgt_kaggle_comparison.csv"; mdp=out/"hgt_kaggle_comparison.md"
    if rows:
        with csvp.open("w",newline="",encoding="utf-8") as f: w=csv.DictWriter(f,fieldnames=list(rows[0])); w.writeheader(); w.writerows(rows)
        mdp.write_text("| "+" | ".join(rows[0])+" |\n| "+" | ".join(["---"]*len(rows[0]))+" |\n"+"\n".join("| "+" | ".join(str(x) for x in row.values())+" |" for row in rows), encoding="utf-8")
    return rows

def bundle():
    if RESULT_ZIP.exists(): RESULT_ZIP.unlink()
    roots=[WORK_DIR/"outputs", GRAPH_META_JSON, GRAPH_STORE_ROOT/"manifest.json", WORK_DIR/"configs/hgt_t082_k5_l3_d01.yaml", WORK_DIR/"configs/hgt_paper_variants"]
    with zipfile.ZipFile(RESULT_ZIP,"w",compression=zipfile.ZIP_STORED,allowZip64=True) as z:
        for root in roots:
            if not root.exists(): continue
            if root.is_file(): z.write(root, root.relative_to(WORK_DIR) if str(root).startswith(str(WORK_DIR)) else root.name); continue
            for p in root.rglob("*"):
                if p.is_file(): z.write(p,p.relative_to(WORK_DIR))
    print("Bundle:", RESULT_ZIP, "MB", round(RESULT_ZIP.stat().st_size/1024/1024,2))

rows=build_comparison(); bundle(); rows


## Notes

- Rerun is safe: each stage skips when its output already exists.
- Full production retraining uses `PIPELINE_MODE = "full_from_pcap"`.
- For old experiments where graph NPZ already exists, set `PIPELINE_MODE = "existing_graph_npz"`.
- `graph_store_v1` may be runtime JSONL. HGT large-graph training requires `layout = numpy_memmap_csr`, written here as `graph_store_mmap_csr_v1`.
- Two GPUs do not merge into one 30GB GPU. `paper_variants` uses both GPUs by training independent configs in parallel. `deployment` trains one production config, so it normally uses one GPU.
